# Customer Feedback Analysis and Automated Response

## Imarticus Data Science Internship Assessment

### Objective
Develop a Python-based solution to:
- Clean customer review data.
- Identify critical customer reviews using rule-based logic.
- Generate personalized apology emails using Generative AI.

In [1]:
import pandas as pd


In [2]:
import re

## Step 1: Load Dataset

In [3]:
df=pd.read_excel("reviews.xlsx")

## exploring data

In [4]:
df.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 568454 entries, 0 to 568453
Data columns (total 10 columns):
 #   Column                  Non-Null Count   Dtype 
---  ------                  --------------   ----- 
 0   Id                      568454 non-null  int64 
 1   ProductId               568454 non-null  object
 2   UserId                  568454 non-null  object
 3   ProfileName             568358 non-null  object
 4   HelpfulnessNumerator    568454 non-null  int64 
 5   HelpfulnessDenominator  568454 non-null  int64 
 6   Score                   568454 non-null  int64 
 7   Time                    568454 non-null  int64 
 8   Summary                 568417 non-null  object
 9   Text                    568454 non-null  object
dtypes: int64(5), object(5)
memory usage: 43.4+ MB


In [6]:
df.columns

Index(['Id', 'ProductId', 'UserId', 'ProfileName', 'HelpfulnessNumerator',
       'HelpfulnessDenominator', 'Score', 'Time', 'Summary', 'Text'],
      dtype='object')

In [7]:
df.describe()

,Id,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time
count,568454.000000,568454.000000,568454.00000,568454.000000,5.684540e+05
mean,284227.500000,1.743817,2.22881,4.183199,1.296257e+09
std,164098.679298,7.636513,8.28974,1.310436,4.804331e+07
min,1.000000,0.000000,0.00000,1.000000,9.393408e+08
25%,142114.250000,0.000000,0.00000,4.000000,1.271290e+09
50%,284227.500000,0.000000,1.00000,5.000000,1.311120e+09
75%,426340.750000,2.000000,2.00000,5.000000,1.332720e+09
max,568454.000000,866.000000,923.00000,5.000000,1.351210e+09


## data cleaning

In [8]:
df.isnull().sum()

Id                         0
ProductId                  0
UserId                     0
ProfileName               96
HelpfulnessNumerator       0
HelpfulnessDenominator     0
Score                      0
Time                       0
Summary                   37
Text                       0
dtype: int64

In [9]:
df.duplicated().sum()

np.int64(0)

In [10]:
reviews_df = df[['ProfileName', 'Score', 'Summary', 'Text']].copy()

In [11]:
reviews_df.isnull().sum()

ProfileName    96
Score           0
Summary        37
Text            0
dtype: int64

In [12]:
reviews_df["ProfileName"]=reviews_df["ProfileName"].fillna("Customer")
reviews_df["Summary"]=reviews_df["Summary"].fillna("No Summary")

In [13]:
reviews_df.isnull().sum()

ProfileName    0
Score          0
Summary        0
Text           0
dtype: int64

### Convert review text to lowercase

In [14]:
reviews_df["Text"] = reviews_df["Text"].str.lower()

In [15]:
reviews_df["Text"].head()

0    i have bought several of the vitality canned d...
1    product arrived labeled as jumbo salted peanut...
2    this is a confection that has been around a fe...
3    if you are looking for the secret ingredient i...
4    great taffy at a great price.  there was a wid...
Name: Text, dtype: object

### Remove special characters

In [16]:
import re

reviews_df["Text"] = reviews_df["Text"].apply(
    lambda x: re.sub(r"[^a-zA-Z0-9\s]", "", x)
)

In [17]:
reviews_df["Text"].head()

0    i have bought several of the vitality canned d...
1    product arrived labeled as jumbo salted peanut...
2    this is a confection that has been around a fe...
3    if you are looking for the secret ingredient i...
4    great taffy at a great price  there was a wide...
Name: Text, dtype: object

In [18]:
reviews_df[reviews_df["Text"] == ""]

,ProfileName,Score,Summary,Text


## Rule-Based Filtering
### Filter reviews with rating 1 or 2

In [19]:
critical_reviews = reviews_df[reviews_df["Score"] <= 2]
len(critical_reviews)

82037

In [20]:
critical_reviews.shape

(82037, 4)

## Step 6: Verify Complaint Detection (True/False)
Check whether each critical review contains complaint keywords using the custom `has_complaint()` function.

In [21]:
complaint_keywords = [
    "broken",
    "late",
    "refund",
    "damaged",
    "rude",
    "poor",
    "worst",
    "bad",
    "terrible",
    "missing",
    "defective",
    "disappointed"
]

In [22]:
def has_complaint(review):
    for keyword in complaint_keywords:
        if keyword in review:
            return True
    return False

In [23]:
critical_reviews["Complaint"] = critical_reviews["Text"].apply(has_complaint)

C:\Users\DELL\AppData\Local\Temp\ipykernel_32004\1247349932.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  critical_reviews["Complaint"] = critical_reviews["Text"].apply(has_complaint)


In [24]:
critical_reviews.head(20)

,ProfileName,Score,Summary,Text,Complaint
1,dll pa,1,Not as Advertised,product arrived labeled as jumbo salted peanut...,False
3,Karl,2,Cough Medicine,if you are looking for the secret ingredient i...,False
12,LT,1,My Cats Are Not Fans of the New Food,my cats have been happily eating felidae plati...,True
16,Erica Neathery,2,poor taste,i love eating them and they are good for watch...,False
26,lady21,1,Nasty No flavor,the candy is just red no flavor just plan a...,False
50,Roberto A,1,Don't like it,this oatmeal is not good its mushy soft i dont...,False
62,gary sturrock,1,stale product.,arrived in 6 days and were so stale i could no...,False
67,Bill Shirer,2,Taste is not so good.,i purchased the mango flavor and to me it does...,True
73,Diana Robinson,1,Warning! WARNING! -ALCOHOL SUGARS!,buyer beware please this sweetener is not for ...,True
74,Jen,2,nothing special,it is okay i would not go out of my way to bu...,False


In [25]:
final_reviews = critical_reviews[critical_reviews["Complaint"] == True]

In [26]:
final_reviews

,ProfileName,Score,Summary,Text,Complaint
12,LT,1,My Cats Are Not Fans of the New Food,my cats have been happily eating felidae plati...,True
67,Bill Shirer,2,Taste is not so good.,i purchased the mango flavor and to me it does...,True
73,Diana Robinson,1,Warning! WARNING! -ALCOHOL SUGARS!,buyer beware please this sweetener is not for ...,True
99,Melissa Benjamin,1,Bad,i fed this to my golden retriever and he hated...,True
146,Barbie,2,BROKEN BOTTLE BOTTOMS!,the salsa smelled delicious as i think it prob...,True
...,...,...,...,...,...
568366,Shosh,2,Missing even a hint of bergamot,i am very disappointed after drinking my first...,True
568402,msfreixy,1,alternative sweetner,i was disappointed in this product as i had re...,True
568434,Katherine Kelly,2,Not so good,this soup is mostly broth although it has a ki...,True
568446,Andy,2,Mixed wrong,i had ordered some of these a few months back ...,True


In [27]:
len(final_reviews)

29647

In [28]:
top_3_reviews = final_reviews.head(3)
top_3_reviews

,ProfileName,Score,Summary,Text,Complaint
12,LT,1,My Cats Are Not Fans of the New Food,my cats have been happily eating felidae plati...,True
67,Bill Shirer,2,Taste is not so good.,i purchased the mango flavor and to me it does...,True
73,Diana Robinson,1,Warning! WARNING! -ALCOHOL SUGARS!,buyer beware please this sweetener is not for ...,True


In [29]:
for review in top_3_reviews["Text"]:
    print(review)
    print("-" * 80)

my cats have been happily eating felidae platinum for more than two years i just got a new bag and the shape of the food is different they tried the new food when i first put it in their bowls and now the bowls sit full and the kitties will not touch the food ive noticed similar reviews related to formula changes in the past unfortunately i now need to find a new food that my cats will eat
--------------------------------------------------------------------------------
i purchased the mango flavor and to me it doesnt take like mango at all  there is no hint of sweetness and unfortunately there is a hint or aftertaste almost like licorice  ive been consuming various sports nutrition products for decades so im familiar and have come to like the taste of the most of the products ive tried  the mango flavor is one of the least appealing ive tasted  its not terrible but its bad enough that i notice the bad taste every sip i take
--------------------------------------------------------------

## Step 6: Generate AI Apology Emails

In [ ]:
!pip install -q google-generativeai



[notice] A new release of pip is available: 24.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [34]:
import google.generativeai as genai

In [ ]:
genai.configure(api_key="YOUR_API_KEY_HERE")

In [36]:
model = genai.GenerativeModel("models/gemini-flash-latest")

In [71]:
review = top_3_reviews.iloc[0]["Text"]

In [69]:
prompt = f"""
You are a Customer Support Agent.

A customer wrote the following review:

{review}

Write a professional, empathetic apology email addressing the customer's complaint.
"""

In [72]:
response = model.generate_content(prompt)

In [40]:
print(response.text)

Subject: Sincerely apologizing for your recent experience with Felidae Platinum

Dear [Customer Name],

Thank you for reaching out to us, and please accept our sincerest apologies for the experience you’ve had with your recent purchase of Felidae Platinum. 

Having your cats happily enjoy our food for more than two years is something we have been so proud of, and I am incredibly sorry that this recent bag has disrupted their routine. As a pet parent, I know how stressful and frustrating it is when your cats suddenly refuse their food, and how daunting it can be to have to search for a new brand that they will accept. 

We did recently make updates to our kibble shape and formula, and while these changes are made with the goal of maintaining high-quality nutrition, we know that cats are incredibly sensitive to even the slightest variations in texture, aroma, and shape. 

Your feedback is extremely important to us, especially given your long-term loyalty. I have shared your comments dire

In [73]:
emails = []

for review in top_3_reviews["Text"]:

    prompt = f"""
    You are a Customer Support Agent.

    Customer Review:
    {review}

    Write a professional, empathetic apology email.
    """

    response = model.generate_content(prompt)
    emails.append(response.text)


In [67]:
top_3_reviews

,ProfileName,Score,Summary,Text,Complaint
12,LT,1,My Cats Are Not Fans of the New Food,my cats have been happily eating felidae plati...,True
67,Bill Shirer,2,Taste is not so good.,i purchased the mango flavor and to me it does...,True
73,Diana Robinson,1,Warning! WARNING! -ALCOHOL SUGARS!,buyer beware please this sweetener is not for ...,True


In [76]:

top_3_reviews["Apology_Email"] = emails

C:\Users\DELL\AppData\Local\Temp\ipykernel_32004\619946560.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  top_3_reviews["Apology_Email"] = emails


In [78]:
top_3_reviews[["ProfileName", "Text", "Apology_Email"]]

,ProfileName,Text,Apology_Email
12,LT,my cats have been happily eating felidae plati...,Subject: Regarding your experience with Felida...
67,Bill Shirer,i purchased the mango flavor and to me it does...,Subject: Your feedback on our Mango flavor - L...
73,Diana Robinson,buyer beware please this sweetener is not for ...,Subject: Sincere apologies regarding your expe...


In [79]:
top_3_reviews.to_excel("Generated_Apology_Emails.xlsx", index=False)

# Conclusion

This project demonstrates how customer feedback can be analyzed using Python and Pandas, and how Generative AI (Google Gemini API) can automatically generate professional apology emails for critical customer reviews.

### Key Steps
- Loaded and explored customer review data.
- Cleaned missing values.
- Identified complaint reviews using rule-based filtering.
- Selected the top 3 critical reviews.
- Used the Google Gemini API to generate empathetic apology emails.
- Stored the generated emails in the dataset.

This workflow shows how AI can assist customer support teams by automating responses while maintaining a professional and empathetic tone.